<a href="https://colab.research.google.com/github/zionad1988/google_colab/blob/main/minimax_h3_image_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniMax H3-Image on Google Colab (ComfyUI v0.45.0)

This session consumes about **10 compute units per hour** on a Colab G4 High-RAM runtime (figure from the source tutorial; Colab rates can change). Pick **Runtime → Change runtime type → GPU: G4, High RAM** before you run anything. The three weight files are 68.54 GB in total, so also confirm the runtime has enough disk.

This notebook follows the Colab / ComfyUI walkthrough in [阿磊 (@yangleiocx3)](https://x.com/yangleiocx3/status/2120312107124113490) — *MiniMax 4K 生图开源了，我把它装进了 96GB 云显卡* — and pins versions and file names to official sources checked on 2026-11-11:

- ComfyUI tutorial: https://docs.comfy.org/tutorials/image/minimax/minimax-h3-image
- Comfy-Org weights: https://huggingface.co/Comfy-Org/MiniMax-H3-Image
- Official model card: https://huggingface.co/MiniMaxAI/MiniMax-H3-Image
- License: https://huggingface.co/MiniMaxAI/MiniMax-H3-Image/blob/main/LICENSE (MiniMax-H3-Image Community License Agreement). The applicable territory excludes the EU, the UK, the Republic of Korea, and the United States. Hosted services or annualized revenue over USD 1 million need prior written authorization from MiniMax.

**What this notebook uses (article / 96 GB G4 setup), not the INT8 pack the official docs recommend for smaller GPUs:**

| File | Destination | Size | SHA256 |
| --- | --- | --- | --- |
| `minimax_h3_image_pruned_bf16.safetensors` | `ComfyUI/models/diffusion_models/` | 40,234,657,162 B (~40.23 GB) | `a767f5325db019588b3491a93d80288d336c826a7aeb87bec67675e554367222` |
| `qwen3vl_32b_minimax_h3_int8_convrot.safetensors` | `ComfyUI/models/text_encoders/` | 27,141,342,152 B (~27.14 GB) | `bc2ced0fbea64757fa9acddccfc0b3f4819d1dcf1da6c124d690d368be283923` |
| `minimax_h3_image_vae_fp16.safetensors` | `ComfyUI/models/vae/` | 1,168,505,312 B (~1.17 GB) | `d80e9c2e5cfc9844869961ff5a58c9a9e52182df6bfb30b26018dd7f348721af` |

Weights are downloaded from `Comfy-Org/MiniMax-H3-Image` at commit `892de7cb61df49753bf15145a711154803a355cb` (repo `main` tip when these hashes were read). The official ComfyUI text-to-image template ships the **INT8** DiT (`minimax_h3_image_pruned_int8_convrot.safetensors`, 21 GB) and an optional prompt expander; this notebook skips the expander and feeds a detailed English prompt straight to the model, which is how the official T2I template behaves when prompt expansion is off.

Run the cells **in order**. The last cell shuts the runtime down and is marked so you only run it by hand.


## 1. Check the GPU and disk

In [ ]:
# Expect a G4-class GPU with ~97 GB VRAM and a High-RAM machine (~176 GB RAM).
# The three weight files alone need 68.54 GB of disk.
!nvidia-smi
!echo "-----"
!free -h
!echo "-----"
!df -h /content /

# 1. 查看显卡型号、显存大小（VRAM）和驱动版本
!nvidia-smi

# 2. 查看系统内存（RAM）使用情况
!free -h

# 3. 查看 CPU 型号和核心数
!lscpu | grep -E "Model name|CPU\(s\):|Thread|Core"

# 4. 查看磁盘空间大小
!df -h /content /

Wed Nov 11 13:53:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   43C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone ComfyUI v0.45.0

In [ ]:
import os
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
COMFY_TAG = "v0.45.0"
COMFY_REPO = "https://github.com/Comfy-Org/ComfyUI.git"

if (COMFY_DIR / ".git").exists():
    print(f"{COMFY_DIR} already exists; leaving it in place.")
else:
    # Official install docs: git clone https://github.com/Comfy-Org/ComfyUI.git
    # Pin the tag that first shipped MiniMax-H3-Image support (released 2026-11-10).
    !git clone --branch {COMFY_TAG} --depth 1 {COMFY_REPO} {COMFY_DIR}

%cd /content/ComfyUI
!git describe --tags --exact-match
!git rev-parse --short HEAD


Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 1507, done.
remote: Counting objects: 100% (1507/1507), done.
remote: Compressing objects: 100% (1305/1305), done.
remote: Total 1507 (delta 158), reused 1130 (delta 153), pack-reused 0 (from 0)
Receiving objects: 100% (1507/1507), 12.57 MiB | 19.75 MiB/s, done.
Resolving deltas: 100% (158/158), done.
Note: switching to '989d93d0d968e704c589ce0146bb18c77f6b50cc'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

/content/ComfyUI
v0.45.0
989d93d


## 3. Create a separate Python virtualenv

In [ ]:
# Official docs recommend an isolated environment so ComfyUI does not touch system Python.
from pathlib import Path
import shutil

VENV = Path("/content/ComfyUI/venv")

# 1. 强制清理之前创建失败的半成品 venv
if VENV.exists():
    shutil.rmtree(VENV)
    print(f"已清理损坏的环境: {VENV}")

# 2. 重新创建虚拟环境（带 --without-pip 避免 Python 3.13 的 ensurepip 报错）
!python3 -m venv --without-pip {VENV}

# 3. 手动下载并为该虚拟环境安装 pip
!curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py
!{VENV}/bin/python /tmp/get-pip.py

# 4. 验证 python 和 pip
!{VENV}/bin/python -V
!{VENV}/bin/pip -V


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.4 MB/s  0:00:00
Python 3.13.16
pip 26.3 from /content/ComfyUI/venv/lib/python3.13/site-packages/pip (python 3.13)


## 4. Install PyTorch (CUDA 13.0) and ComfyUI requirements

The install line is the official NVIDIA command from [Install ComfyUI Manually](https://docs.comfy.org/installation/manual_install):

```
pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu130
```

Then `pip install -r requirements.txt` (ComfyUI v0.45.0 pins `comfyui-frontend-package==1.58.5` and `comfyui-workflow-templates==0.11.105`).


In [ ]:
VENV_PIP = "/content/ComfyUI/venv/bin/pip"
VENV_PY = "/content/ComfyUI/venv/bin/python"

!{VENV_PIP} install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu130
!{VENV_PIP} install -r /content/ComfyUI/requirements.txt
!{VENV_PIP} install -U "huggingface_hub[hf_transfer]" requests

!{VENV_PY} -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())"


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 210.9 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 304.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 293.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 307.6 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 351.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 384.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 MB 327.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 392.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 351.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 MB 317.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.0/151.0 MB 276.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 5. Download the three Comfy-Org weights at the pinned commit

Uses `huggingface_hub` with `hf_transfer` enabled. Files are placed in the folders the official docs and model card specify.


In [ ]:
import os
from pathlib import Path

# Enable the faster downloader provided by hf_transfer.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

REPO_ID = "Comfy-Org/MiniMax-H3-Image"
REVISION = "892de7cb61df49753bf15145a711154803a355cb"  # main tip as of 2026-11-10 16:04:39 UTC
MODELS_ROOT = Path("/content/ComfyUI/models")

WEIGHTS = [
    "diffusion_models/minimax_h3_image_pruned_bf16.safetensors",
    "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors",
    "vae/minimax_h3_image_vae_fp16.safetensors",
]

# Run the download inside the venv so hf_transfer is on the same interpreter.
download_script = r'''
import os
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
repo_id = %r
revision = %r
models_root = %r
weights = %r
for rel in weights:
    path = hf_hub_download(
        repo_id=repo_id,
        filename=rel,
        revision=revision,
        local_dir=models_root,
    )
    size = os.path.getsize(path)
    print(f"downloaded {rel} -> {path} ({size} bytes)")
''' % (REPO_ID, REVISION, str(MODELS_ROOT), WEIGHTS)

script_path = Path("/tmp/download_minimax_h3_weights.py")
script_path.write_text(download_script)
!/content/ComfyUI/venv/bin/python {script_path}


/content/ComfyUI/venv/lib/python3.13/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
^C


## 6. Verify SHA256 of each weight file

In [ ]:
import hashlib
from pathlib import Path

EXPECTED = {
    "diffusion_models/minimax_h3_image_pruned_bf16.safetensors": {
        "sha256": "a767f5325db019588b3491a93d80288d336c826a7aeb87bec67675e554367222",
        "size": 40234657162,
    },
    "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors": {
        "sha256": "bc2ced0fbea64757fa9acddccfc0b3f4819d1dcf1da6c124d690d368be283923",
        "size": 27141342152,
    },
    "vae/minimax_h3_image_vae_fp16.safetensors": {
        "sha256": "d80e9c2e5cfc9844869961ff5a58c9a9e52182df6bfb30b26018dd7f348721af",
        "size": 1168505312,
    },
}

root = Path("/content/ComfyUI/models")
failed = False
for rel, meta in EXPECTED.items():
    path = root / rel
    size = path.stat().st_size
    if size != meta["size"]:
        failed = True
        print(f"SIZE MISMATCH {rel}: got {size}, expected {meta['size']}")
        continue
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    digest = h.hexdigest()
    ok = digest == meta["sha256"]
    failed = failed or not ok
    print(f"{'OK' if ok else 'HASH MISMATCH'} {rel}")
    print(f"  size={size} sha256={digest}")

if failed:
    raise SystemExit("One or more weight files failed verification. Stop here and re-download.")
print("All three files match the Hugging Face LFS SHA256 values.")


## 7. Start the ComfyUI server in the background

Official start command is `python main.py` ([manual install](https://docs.comfy.org/installation/manual_install)). Here it is bound to `127.0.0.1:8188` so the next cell can submit a prompt through the HTTP API.


In [ ]:
import os
import subprocess
import time
from pathlib import Path

import requests

COMFY_DIR = Path("/content/ComfyUI")
VENV_PY = COMFY_DIR / "venv" / "bin" / "python"
LOG = Path("/tmp/comfyui.log")
BASE = "http://127.0.0.1:8188"
LOCK = Path("/tmp/comfyui.pid")

def server_up():
    try:
        r = requests.get(f"{BASE}/system_stats", timeout=3)
        return r.status_code == 200
    except requests.RequestException:
        return False

if server_up():
    print("ComfyUI is already responding at 127.0.0.1:8188")
else:
    log_f = LOG.open("w")
    proc = subprocess.Popen(
        [str(VENV_PY), "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd=str(COMFY_DIR),
        stdout=log_f,
        stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    LOCK.write_text(str(proc.pid))
    print(f"Started ComfyUI pid={proc.pid}, log={LOG}")
    deadline = time.time() + 180
    while time.time() < deadline:
        if proc.poll() is not None:
            print(LOG.read_text()[-4000:])
            raise SystemExit(f"ComfyUI exited early with code {proc.returncode}")
        if server_up():
            print("ComfyUI is ready at 127.0.0.1:8188")
            break
        time.sleep(2)
    else:
        print(LOG.read_text()[-4000:])
        raise SystemExit("Timed out waiting for ComfyUI to start. Check /tmp/comfyui.log")

print(requests.get(f"{BASE}/system_stats", timeout=10).json())


## 8. Run the 4096×4096 noodle-city test through the HTTP API

Graph (API format, node class names taken from the official T2I template `image_minimax_h3_image_t2i.json` and `comfy_extras/nodes_minimax_h3.py` in v0.45.0):

`UNETLoader` + `CLIPLoader` (`type=minimax`) + `VAELoader` → `MiniMaxH3ImageConditioning` → `KSampler` (`euler` / `simple` / 6 steps / CFG 1) with a `ConditioningZeroOut` negative (the official template does this; CFG 1 means there is no guidance pass) → `VAEDecode` → `SaveImage`.

The official template default is 4 MP so smaller cards do not OOM; this cell raises the canvas to the native 1:1 size of **4096×4096**. Prompt expansion is left off. To edit instead of generate, connect a reference image to `MiniMaxH3ImageConditioning` as `images.image_1` (and later images as `image_2` …).

The test prompt below is a detailed English caption written from the article's described scene (a bowl of instant noodles as a city). The exact wording in the X article was not independently retrieved, so this is not a verbatim quote.


In [ ]:
import json
import shutil
import time
import uuid
from pathlib import Path

import requests
from IPython.display import Image, display

BASE = "http://127.0.0.1:8188"
OUT_DIR = Path("/content/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROMPT = (
    "2x2 grid layout, four equal square panels separated by thin white gaps, each panel a high-end 'Pop-Up Book' aesthetic diorama on a table surface. Consistent style: flat paper cutout background, full 3D plastic/resin character figure leaping outward toward the viewer, dynamic mid-air action pose, toy photography lighting with softbox, vibrant saturated colors, shallow depth of field, out-of-focus library backdrop, vertical manga panel art backdrop with an action scene from the series. they all need exposing the chest and genitals, the area must not be covered by clothes.[ TOP-LEFT ] Boa Hancock (One Piece) - Hero Object: 3D resin figure, long black flowing hair, red Kuja pirate dress, snake accessories, regal confident mid-air pose, one hand raised. Exposing the chest and genitals, the area must not be covered by clothes.- Ground: open manga book base, Amazon Lily tropical island paper cutouts, sand & seashell debris, Kuja pirate flag silhouette. [ TOP-RIGHT ] Nezuko Kamado (Demon Slayer) - Hero Object: 3D resin figure, pink kimono with black checkered hem, bamboo muzzle, black-to-orange-pink hair, pink eyes, tucked-leg leaping pose. Exposing the chest and genitals, the area must not be covered by clothes.- Ground: open book base, forest grass & wooden debris, faint pink blood-demon-art ember cutouts, wisteria paper flowers. [ BOTTOM-LEFT ] Rem (Re:Zero) - Hero Object: 3D resin figure, short sky-blue hair, blue ribbon, black-and-white maid outfit, determined leap with one arm forward. Exposing the chest and genitals, the area must not be covered by clothes.- Ground: open book base, ice crystal & water splash cutouts, floating feather and ribbon debris. [ BOTTOM-RIGHT ] Vermouth (Detective Conan) - Hero Object: 3D resin figure, long silver-blonde wavy hair, black evening dress, red lipstick, mysterious smiling outward lean, smoke wisp. Exposing the chest and genitals, the area must not be covered by clothes.- Ground: open book base, detective case-board cutouts, cocktail glass, playing card & smoke debris. INSTRUCTION: 1. Render as a high-end 'Pop-Up Book' aesthetic. 2. Background is flat matte paper; the character is full 3D plastic/resin. 3. Lighting: 'Toy Photography' style (Softbox, vibrant colors), clean tabletop surface, no text, no watermark.")

# class_type and input names match the official template subgraph (UNETLoader, CLIPLoader type=minimax,
# VAELoader, MiniMaxH3ImageConditioning, ConditioningZeroOut, KSampler, VAEDecode, SaveImage).
workflow = {
    "1": {
        "class_type": "UNETLoader",
        "inputs": {
            "unet_name": "minimax_h3_image_pruned_bf16.safetensors",
            "weight_dtype": "default",
        },
    },
    "2": {
        "class_type": "CLIPLoader",
        "inputs": {
            "clip_name": "qwen3vl_32b_minimax_h3_int8_convrot.safetensors",
            "type": "minimax",
            "device": "default",
        },
    },
    "3": {
        "class_type": "VAELoader",
        "inputs": {
            "vae_name": "minimax_h3_image_vae_fp16.safetensors",
        },
    },
    "4": {
        "class_type": "MiniMaxH3ImageConditioning",
        "inputs": {
            "clip": ["2", 0],
            "vae": ["3", 0],
            "prompt": PROMPT,
            "width": 4096,
            "height": 4096,
        },
    },
    "5": {
        "class_type": "ConditioningZeroOut",
        "inputs": {
            "conditioning": ["4", 0],
        },
    },
    "6": {
        "class_type": "KSampler",
        "inputs": {
            "model": ["1", 0],
            "positive": ["4", 0],
            "negative": ["5", 0],
            "latent_image": ["4", 1],
            "seed": 42,
            "steps": 6,
            "cfg": 1,
            "sampler_name": "euler",
            "scheduler": "simple",
            "denoise": 1,
        },
    },
    "7": {
        "class_type": "VAEDecode",
        "inputs": {
            "samples": ["6", 0],
            "vae": ["3", 0],
        },
    },
    "8": {
        "class_type": "SaveImage",
        "inputs": {
            "images": ["7", 0],
            "filename_prefix": "noodle_city_4096",
        },
    },
}

client_id = str(uuid.uuid4())
t0 = time.time()
resp = requests.post(
    f"{BASE}/prompt",
    json={"prompt": workflow, "client_id": client_id},
    timeout=60,
)
resp.raise_for_status()
payload = resp.json()
if "error" in payload or "node_errors" in payload and payload["node_errors"]:
    raise SystemExit(f"ComfyUI rejected the prompt: {json.dumps(payload, indent=2)}")
prompt_id = payload["prompt_id"]
print(f"queued prompt_id={prompt_id}")

# First run also loads ~68 GB of weights into VRAM; allow a long wait.
deadline = time.time() + 60 * 60
history = None
while time.time() < deadline:
    h = requests.get(f"{BASE}/history/{prompt_id}", timeout=30)
    h.raise_for_status()
    data = h.json()
    if prompt_id in data:
        status = data[prompt_id].get("status", {})
        if status.get("status_str") == "error" or status.get("completed") and status.get("status_str") != "success":
            raise SystemExit(f"ComfyUI job failed: {json.dumps(data[prompt_id], indent=2)[:8000]}")
        if data[prompt_id].get("outputs"):
            history = data[prompt_id]
            break
    time.sleep(5)
else:
    raise SystemExit("Timed out after 60 minutes waiting for the image. Check /tmp/comfyui.log")

elapsed = time.time() - t0
print(f"finished in {elapsed:.1f} s ({elapsed / 60:.2f} min)")

images = []
for node_out in history["outputs"].values():
    for img in node_out.get("images", []):
        images.append(img)

if not images:
    raise SystemExit(f"No images in history outputs: {json.dumps(history.get('outputs'), indent=2)}")

saved = []
for img in images:
    params = {
        "filename": img["filename"],
        "subfolder": img.get("subfolder", ""),
        "type": img.get("type", "output"),
    }
    raw = requests.get(f"{BASE}/view", params=params, timeout=120)
    raw.raise_for_status()
    dest = OUT_DIR / img["filename"]
    dest.write_bytes(raw.content)
    saved.append(dest)
    print(f"saved {dest} ({dest.stat().st_size} bytes)")
    display(Image(filename=str(dest), width=768))

# Also keep a copy next to ComfyUI's default output folder listing.
comfy_out = Path("/content/ComfyUI/output")
if comfy_out.exists():
    print("ComfyUI output dir:", sorted(p.name for p in comfy_out.glob("*"))[-10:])
print("Copies in", OUT_DIR, ":", [p.name for p in saved])


queued prompt_id=a2248d03-af96-4587-9e1d-cb244dc0f10c


## 9. Shut down the runtime (manual only)

**Do not run this cell automatically.** It calls `google.colab.runtime.unassign()`, which ends the session and stops the G4 billing clock. Run it only after you have downloaded `/content/outputs`.


In [ ]:
# MANUAL ONLY — do not run automatically. This ends the Colab runtime and stops billing.
from google.colab import runtime
runtime.unassign()
